# PSFMachine en TESS (Notebook)

Este notebook procesa una lista de TIC IDs de TESS con `lightkurve` + `psfmachine` y guarda curvas de luz por objetivo.

In [ ]:
%pip install -q -U pip
%pip install -q lightkurve psfmachine pandas numpy

In [ ]:
from pathlib import Path
import json
import lightkurve as lk
import psfmachine as pm
import pandas as pd

In [ ]:
TICS = [
    11300044, 199780530, 264459850, 427395094, 427395136,
    200093870, 138829413, 200093884, 269118295, 264462165,
    264462237, 284284106, 284284069, 427347825, 427347969,
    427393284, 427394530, 427395536, 50897998, 712930938
]
SECTORS = None  # None = all, o por ejemplo [70,71]
CADENCE = 'short'  # 'short', 'long', 'fast'
OUTPUT_DIR = Path('outputs_notebook')
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

In [ ]:
def export_lightcurves(lcs, outdir: Path):
    exported = 0
    for idx, lc in enumerate(lcs):
        df = lc.to_pandas()
        if df.empty:
            continue
        df.to_csv(outdir / f'lc_{idx:03d}.csv')
        exported += 1
    return exported

def process_tic(tic: int, sectors=None, cadence='short', output_dir=Path('outputs_notebook')):
    tic_dir = output_dir / f'TIC_{tic}'
    tic_dir.mkdir(parents=True, exist_ok=True)

    query = lk.search_targetpixelfile(
        f'TIC {tic}',
        mission='TESS',
        cadence=cadence,
        sector=sectors,
    )
    if len(query) == 0:
        raise RuntimeError('No se encontraron TPFs para el criterio dado')

    tpfs = query.download_all()
    if tpfs is None or len(tpfs) == 0:
        raise RuntimeError('No se pudieron descargar TPFs')

    machine = pm.TPFMachine.from_TPFs(tpfs)
    machine.fit_lightcurves(sap=True)

    exported = export_lightcurves(machine.lcs, tic_dir)
    log = {
        'tic': tic,
        'n_tpfs': len(tpfs),
        'n_lcs_exportadas': exported,
        'cadence': cadence,
        'sectors': sectors if sectors is not None else 'all',
    }
    (tic_dir / 'log.json').write_text(json.dumps(log, indent=2, ensure_ascii=False))
    return log

In [ ]:
results = []
for tic in TICS:
    try:
        log = process_tic(tic, sectors=SECTORS, cadence=CADENCE, output_dir=OUTPUT_DIR)
        results.append({'tic': tic, 'status': 'ok', **log})
        print(f'[OK] TIC {tic}')
    except Exception as exc:
        err_dir = OUTPUT_DIR / f'TIC_{tic}'
        err_dir.mkdir(parents=True, exist_ok=True)
        (err_dir / 'error.txt').write_text(str(exc))
        results.append({'tic': tic, 'status': 'error', 'error': str(exc)})
        print(f'[ERROR] TIC {tic}: {exc}')

In [ ]:
pd.DataFrame(results)